# Validação do Merge — ETL Telco

Este notebook valida visualmente a correção do bug no merge das 5 bases Telco, antes de a lógica virar o módulo `src/etl/transform.py`.

**Objetivo**: confirmar que os merges são 1:1/many-to-one (sem explosão de linhas), sem colunas duplicadas (`_x`/`_y`), e que o merge com `populations` usa a chave correta (`on=zip_code` em vez de `left_on="locations_zip_code"`).

In [17]:
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
BASE_URL = "https://public.dhe.ibm.com/software/data/sw-library/cognos/mobile/C11/data/"

ARQUIVOS = [
    "Telco_customer_churn_demographics.xlsx",
    "Telco_customer_churn_location.xlsx",
    "Telco_customer_churn_population.xlsx",
    "Telco_customer_churn_services.xlsx",
    "Telco_customer_churn_status.xlsx",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)

# Baixar arquivos
for nome in ARQUIVOS:
    caminho = RAW_DIR / nome
    if caminho.exists():
        print(f"✓ {nome} (já existe)")
    else:
        print(f"↓ Baixando {nome}...")
        try:
            url = f"{BASE_URL}{nome}"
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as response:
                caminho.write_bytes(response.read())
            print("  ✓ Salvo")
        except urllib.error.URLError as e:
            print(f"  ✗ Erro: {e}")
            raise

✓ Telco_customer_churn_demographics.xlsx (já existe)
✓ Telco_customer_churn_location.xlsx (já existe)
✓ Telco_customer_churn_population.xlsx (já existe)
✓ Telco_customer_churn_services.xlsx (já existe)
✓ Telco_customer_churn_status.xlsx (já existe)


## 1. Preparar diretórios e baixar arquivos brutos

In [18]:
print(f"\nArquivos em {RAW_DIR}:")
for arquivo in sorted(RAW_DIR.glob("*.xlsx")):
    print(f"  - {arquivo.name}")


Arquivos em ..\data\raw:
  - CustomerChurn.xlsx
  - Telco_customer_churn.xlsx
  - Telco_customer_churn_demographics.xlsx
  - Telco_customer_churn_location.xlsx
  - Telco_customer_churn_population.xlsx
  - Telco_customer_churn_services.xlsx
  - Telco_customer_churn_status.xlsx


## 2. Carregar as 5 bases brutas

In [19]:
dfs = {
    "demographics": pd.read_excel(RAW_DIR / "Telco_customer_churn_demographics.xlsx"),
    "locations": pd.read_excel(RAW_DIR / "Telco_customer_churn_location.xlsx"),
    "populations": pd.read_excel(RAW_DIR / "Telco_customer_churn_population.xlsx"),
    "services": pd.read_excel(RAW_DIR / "Telco_customer_churn_services.xlsx"),
    "status": pd.read_excel(RAW_DIR / "Telco_customer_churn_status.xlsx"),
}

for name, df in dfs.items():
    print(f"{name:15} -> {df.shape}")

demographics    -> (7043, 9)
locations       -> (7043, 10)
populations     -> (1671, 3)
services        -> (7043, 31)
status          -> (7043, 12)


## 3. Verificar duplicidade de chaves antes do merge

In [20]:
print("Verificação de duplicidade de chaves:")
print()

if "Customer ID" in dfs["locations"].columns:
    for name in ["demographics", "locations", "services", "status"]:
        df = dfs[name]
        duplicados = df["Customer ID"].duplicated().sum()
        print(f"{name:15} | Customer ID duplicados: {duplicados}")

if "Zip Code" in dfs["populations"].columns:
    duplicados = dfs["populations"]["Zip Code"].duplicated().sum()
    print(f"populations    | Zip Code duplicados: {duplicados}")

Verificação de duplicidade de chaves:

demographics    | Customer ID duplicados: 0
locations       | Customer ID duplicados: 0
services        | Customer ID duplicados: 0
status          | Customer ID duplicados: 0
populations    | Zip Code duplicados: 0


## 4. Definir funções de limpeza e padronização (inline, para validação)

In [21]:
import re

_RE_CARACTERES_ESPECIAIS = re.compile(r"[()[]/]")
_RE_ESPACO_HIFEN = re.compile(r"[\s-]+")


def limpar_nome_coluna(coluna: str) -> str:
    """Normaliza um nome de coluna: remove parênteses/colchetes/barras,
    troca espaços/hífens por underscore e converte para minúsculas."""
    nome = str(coluna).strip()
    nome = _RE_CARACTERES_ESPECIAIS.sub("", nome)
    nome = _RE_ESPACO_HIFEN.sub("_", nome)
    return nome.lower()


def padronizar_colunas(df: pd.DataFrame, prefixo: str, location_key: str) -> pd.DataFrame:
    """Limpa e prefixa as colunas de um DataFrame, preservando as chaves de união
    (customer_id e location_key) sem prefixo."""
    novas_colunas: dict[str, str] = {}
    for col in df.columns:
        col_limpa = limpar_nome_coluna(col)
        if col_limpa in ("customer_id", location_key):
            novas_colunas[col] = col_limpa
        else:
            novas_colunas[col] = f"{prefixo}_{col_limpa}"
    return df.rename(columns=novas_colunas)


print("Funções de limpeza definidas.")

Funções de limpeza definidas.


## 5. Aplicar padronização de colunas em cada base

In [22]:
location_key = limpar_nome_coluna("Zip Code")
print(f"Chave de localização após limpeza: '{location_key}'")
print()

dfs_padronizados = {}
for name, df in dfs.items():
    dfs_padronizados[name] = padronizar_colunas(df, name, location_key)
    cols = list(dfs_padronizados[name].columns[:5])
    print(f"{name:15} | Primeiras colunas: {cols}")

Chave de localização após limpeza: 'zip_code'

demographics    | Primeiras colunas: ['customer_id', 'demographics_count', 'demographics_gender', 'demographics_age', 'demographics_under_30']
locations       | Primeiras colunas: ['locations_location_id', 'customer_id', 'locations_count', 'locations_country', 'locations_state']
populations     | Primeiras colunas: ['populations_id', 'zip_code', 'populations_population']
services        | Primeiras colunas: ['services_service_id', 'customer_id', 'services_count', 'services_quarter', 'services_referred_a_friend']
status          | Primeiras colunas: ['status_status_id', 'customer_id', 'status_count', 'status_quarter', 'status_satisfaction_score']


## 6. Merges passo a passo com validação de contagem de linhas

In [23]:
print("Reproduzindo os merges com validação de shape:")
print()

df_merged = dfs_padronizados["locations"].copy()
linhas_inicial = len(df_merged)
print(f"Início (locations): {len(df_merged)} linhas")

# Merge 1: locations + demographics (customer_id)
df_merged = df_merged.merge(dfs_padronizados["demographics"], on="customer_id", how="left")
explosao = len(df_merged) != linhas_inicial
print(f"Depois de demographics: {len(df_merged)} linhas (explosão? {explosao})")

# Merge 2: + services (customer_id)
df_merged = df_merged.merge(dfs_padronizados["services"], on="customer_id", how="left")
explosao = len(df_merged) != linhas_inicial
print(f"Depois de services: {len(df_merged)} linhas (explosão? {explosao})")

# Merge 3: + status (customer_id)
df_merged = df_merged.merge(dfs_padronizados["status"], on="customer_id", how="left")
explosao = len(df_merged) != linhas_inicial
print(f"Depois de status: {len(df_merged)} linhas (explosão? {explosao})")

print("\nAntes de populations:")

Reproduzindo os merges com validação de shape:

Início (locations): 7043 linhas
Depois de demographics: 7043 linhas (explosão? False)
Depois de services: 7043 linhas (explosão? False)
Depois de status: 7043 linhas (explosão? False)

Antes de populations:


In [24]:
zip_in_merged = [c for c in df_merged.columns if "zip" in c.lower()]
zip_in_pop = [c for c in dfs_padronizados["populations"].columns if "zip" in c.lower()]
print(f"  - Colunas de zip_code em df_merged: {zip_in_merged}")
print(f"  - Colunas de zip_code em populations: {zip_in_pop}")

# Correção: usar on=location_key (zip_code sem prefixo)
df_merged = df_merged.merge(dfs_padronizados["populations"], on=location_key, how="left")
explosao = len(df_merged) != linhas_inicial
print(f"Depois de populations (on='{location_key}'): {len(df_merged)} linhas")
print(f"Explosão detectada? {explosao}")

  - Colunas de zip_code em df_merged: ['zip_code']
  - Colunas de zip_code em populations: ['zip_code']
Depois de populations (on='zip_code'): 7043 linhas
Explosão detectada? False


## 7. Validação de qualidade do resultado final

In [25]:
print("Validações finais:")
print()

# Check 1: Shape esperado
print(f"✓ Shape final: {df_merged.shape}")
ok = len(df_merged) == linhas_inicial
print(f"  Linhas: {len(df_merged)} == {linhas_inicial}? {ok}")

# Check 2: Colunas duplicadas (_x, _y)
duplicadas = [c for c in df_merged.columns if c.endswith(("_x", "_y"))]
txt = duplicadas if duplicadas else "Nenhuma (correto!)"
print(f"✓ Colunas com sufixo _x/_y: {txt}")

Validações finais:

✓ Shape final: (7043, 61)
  Linhas: 7043 == 7043? True
✓ Colunas com sufixo _x/_y: Nenhuma (correto!)


In [26]:
# Check 3: Coluna de zip_code única (não duplicada)
zip_cols = [c for c in df_merged.columns if "zip" in c.lower()]
print(f"✓ Colunas com 'zip': {zip_cols}")

# Check 4: Nulos na coluna de população (clientes cujos zip não bateram)
if "populations_population" in df_merged.columns:
    nulos = df_merged["populations_population"].isna().sum()
    print(f"✓ Nulos em populations_population: {nulos}")

✓ Colunas com 'zip': ['zip_code']
✓ Nulos em populations_population: 0


## 8. Resumo e Conclusão

In [27]:
print("=" * 60)
print("RESUMO DA VALIDAÇÃO")
print("=" * 60)
print()
print("Bases originais:")
for name in ["demographics", "locations", "services", "status", "populations"]:
    print(f"  {name}: {dfs[name].shape}")
print()
print(f"Base final unificada: {df_merged.shape}")
print()
print("Testes passados:")
print("  ✓ Sem explosão de linhas (1:1 / many-to-one)")
print("  ✓ Sem colunas duplicadas (_x/_y)")
print("  ✓ Merge com populations usa on='zip_code' (correção do bug)")
print()
print("A lógica acima pode agora ser extraída para src/etl/transform.py!")

RESUMO DA VALIDAÇÃO

Bases originais:
  demographics: (7043, 9)
  locations: (7043, 10)
  services: (7043, 31)
  status: (7043, 12)
  populations: (1671, 3)

Base final unificada: (7043, 61)

Testes passados:
  ✓ Sem explosão de linhas (1:1 / many-to-one)
  ✓ Sem colunas duplicadas (_x/_y)
  ✓ Merge com populations usa on='zip_code' (correção do bug)

A lógica acima pode agora ser extraída para src/etl/transform.py!
